In [0]:
# =============================================================================
# QUALIDADE DE DADOS - Análise da camada Bronze
# =============================================================================
# Objetivo: Verificar a qualidade dos dados brutos antes das transformações,
# identificando problemas de completude, consistência, unicidade e acurácia.
# Essa análise justifica as decisões de limpeza tomadas na camada Silver.
# =============================================================================

from pyspark.sql import functions as F

# Função auxiliar que analisa qualidade de qualquer tabela
def analisar_qualidade(nome_tabela):
    df = spark.table(f"workspace.default.{nome_tabela}")
    total = df.count()
    
    print(f"\n{'='*60}")
    print(f"TABELA: {nome_tabela} | Total de registros: {total:,}")
    print(f"{'='*60}")
    
    # Completude: verifica nulos por coluna
    print("\n--- COMPLETUDE (valores nulos por coluna) ---")
    for col in df.columns:
        if col.startswith("_"):
            continue  # ignora colunas de metadado
        nulos = df.filter(F.col(col).isNull()).count()
        pct = round((nulos / total) * 100, 2)
        if nulos > 0:
            print(f"  {col}: {nulos:,} nulos ({pct}%)")
    
    # Unicidade: verifica duplicatas na primeira coluna (geralmente o ID)
    id_col = [c for c in df.columns if c.endswith("_id") and not c.startswith("_")]
    if id_col:
        id_col = id_col[0]
        duplicatas = total - df.dropDuplicates([id_col]).count()
        print(f"\n--- UNICIDADE ---")
        print(f"  Duplicatas por {id_col}: {duplicatas:,}")

# Analisa as tabelas Bronze principais
for tabela in [
    "bronze_orders",
    "bronze_order_items", 
    "bronze_order_payments",
    "bronze_products",
    "bronze_customers",
    "bronze_sellers",
    "bronze_category_translation"
]:
    analisar_qualidade(tabela)

In [0]:
# =============================================================================
# QUALIDADE DE DADOS - Verificações de acurácia e consistência
# =============================================================================

# --- ACURÁCIA: valores fora do domínio esperado ---
print("="*60)
print("ACURÁCIA - Valores fora do domínio esperado")
print("="*60)

items = spark.table("workspace.default.bronze_order_items")
payments = spark.table("workspace.default.bronze_order_payments")
orders = spark.table("workspace.default.bronze_orders")

# Preços negativos ou zerados em order_items
preco_invalido = items.filter(F.col("price") <= 0).count()
print(f"\nbronze_order_items:")
print(f"  Registros com preço <= 0: {preco_invalido:,}")

# Frete negativo
frete_invalido = items.filter(F.col("freight_value") < 0).count()
print(f"  Registros com frete negativo: {frete_invalido:,}")

# Pagamentos com valor negativo
pagamento_invalido = payments.filter(F.col("payment_value") < 0).count()
print(f"\nbronze_order_payments:")
print(f"  Pagamentos com valor negativo: {pagamento_invalido:,}")

# Formas de pagamento disponíveis (consistência de categorias)
print(f"\n  Formas de pagamento encontradas:")
payments.groupBy("payment_type").count().orderBy("count", ascending=False).show()

# Status de pedidos disponíveis
print(f"bronze_orders:")
print(f"  Status de pedidos encontrados:")
orders.groupBy("order_status").count().orderBy("count", ascending=False).show()

# --- OUTLIERS: valores extremos em preço ---
print("="*60)
print("OUTLIERS - Estatísticas de preço e frete")
print("="*60)
items.select(
    F.round(F.min("price"), 2).alias("preco_min"),
    F.round(F.avg("price"), 2).alias("preco_medio"),
    F.round(F.max("price"), 2).alias("preco_max"),
    F.round(F.min("freight_value"), 2).alias("frete_min"),
    F.round(F.avg("freight_value"), 2).alias("frete_medio"),
    F.round(F.max("freight_value"), 2).alias("frete_max"),
).show()

In [0]:
# =============================================================================
# QUALIDADE DE DADOS - Resumo consolidado dos problemas encontrados
# =============================================================================
# Esse bloco consolida todos os achados da análise de qualidade e documenta
# as decisões de tratamento aplicadas na camada Silver.
# =============================================================================

print("""
RESUMO DA ANÁLISE DE QUALIDADE DE DADOS
========================================

1. COMPLETUDE
-------------
bronze_orders:
  - order_approved_at: 160 nulos (0.16%)
    Tratamento: mantidos. Pedidos sem aprovação podem ser cancelados ou
    em processamento. Não impacta as análises de padrão de compra.
  - order_delivered_carrier_date: 1.783 nulos (1.79%)
  - order_delivered_customer_date: 2.965 nulos (2.98%)
    Tratamento: mantidos. Pedidos ainda em trânsito ou cancelados
    naturalmente não têm data de entrega.

bronze_products:
  - product_category_name: 610 nulos (1.85%)
    Tratamento: removidos na Silver. Produtos sem categoria não contribuem
    para análises por categoria, que é o foco do objetivo.
  - Demais campos descritivos (nome, descrição, fotos, dimensões):
    610 a 2 nulos. Tratamento: mantidos, pois não são usados na Gold.

bronze_customers, bronze_sellers, bronze_category_translation:
  - Sem valores nulos. Dados completos.

2. UNICIDADE
------------
bronze_order_items: 13.984 duplicatas por order_id
  Comportamento esperado. Um pedido pode ter múltiplos itens,
  portanto a chave única é a combinação order_id + order_item_id,
  não o order_id isolado. Sem problema real.

bronze_order_payments: 4.446 duplicatas por order_id
  Comportamento esperado. Um pedido pode ter múltiplas formas de
  pagamento (ex: cartão + voucher). Tratamento: agregados por order_id
  na construção da tabela Gold.

bronze_orders, bronze_products, bronze_customers, bronze_sellers:
  Sem duplicatas nas chaves primárias.

3. ACURÁCIA
-----------
bronze_order_items:
  - Sem preços negativos ou zerados.
  - Frete mínimo de R$ 0,00: possível para retirada em loja ou promoção.
    Mantido como válido.
  - Preço máximo de R$ 6.735,00: valor alto mas plausível para eletrônicos
    ou itens premium. Não caracteriza erro.

bronze_order_payments:
  - 3 registros com payment_type = "not_defined". Volume irrelevante
    (0.003% do total). Mantidos na Silver sem tratamento especial.
  - Sem valores de pagamento negativos.

4. CONSISTÊNCIA
---------------
  - Estados de clientes e vendedores padronizados para maiúsculo na Silver
    (customer_state, seller_state) para evitar inconsistências como
    "sp" vs "SP".
  - Datas convertidas de string para Timestamp na Silver para permitir
    operações temporais corretas nas análises.

CONCLUSÃO
---------
  O dataset da Olist é de boa qualidade geral. Os principais ajustes
  foram de tipagem (datas como string), padronização de categorias
  (maiúsculas) e remoção de produtos sem categoria. Não foram encontrados
  problemas graves de acurácia ou inconsistência que comprometessem
  as análises de padrão de compra.
""")